### $\textbf{DREAM-Disaster Response Queueing Model}$

$\text{User Manual: }$
[Click here](https://docs.google.com/document/d/12MXNLDTKlQ0HM1NfmpoTbaPHug6XjJ3uvQhEK5yXPkE/preview?rm=minimal)
<!-- $\textbf{Definitions:}$ \\
$\text{System: The whole process the victim undergoes until arrival to AMP}$ \\
$\text{Stages: Casualty Collection Point, Staging Area}$ \\
$\text{Servers: Teams per stage (e.g. First Aid Team in CCP)}$ \\
$\text{Rate: Casualties per unit time (hour)}$ \\

$\textbf{Parameters:}$ \\
$\text{$\lambda = $ Arrival rate (casualties per hour)}$ \\
$\text{$\mu = $ Service rate (casualties per hour)}$ \\
$\text{$\rho = $ Proportion of time (hour) the server is busy}$ \\
$\text{$c = $ Number of servers}$ \\

$\textbf{Performance Metrics:}$ \\
$L_s = \text{Expected number of casualties in the current stage}$ \\
$L_q = \text{Expected number of casualties waiting in the current stage}$ \\
$W_s = \text{Expected total time a casualty spends in the system}$ \\
$W_q = \text{Expected waiting time a casualty spends in the queue before being rendered service}$ -->

In [ ]:
# @title $\text{[A] Database}$

from IPython.display import clear_output, display, HTML
from google.colab import output
import os
import json
import time
import re
import math

DATA_PATH = "/content/casualty_data.json"
SETTINGS_PATH = "/content/settings.json"

class CasualtyCalculator:
  def __init__(self):
    self.data = []
    self.settings = {"clear_screen": True, "CCP_config": [1, 0], "SA_config": [1, 0]}

    self.CCPArrivalRateAverage = None
    self.CCPServiceRateAverage = None
    self.SAServiceRateAverage = None

    self.needs_clear = False
    self.loadSettings()
    self.loadData()

  def loadData(self):
    if os.path.exists(DATA_PATH):
      try:
        with open(DATA_PATH, "r") as f:
          self.data = json.load(f)
        self.computeAverages(False)
        print(f"[System] Loaded {len(self.data)} entries from the previous session.")
      except Exception as e:
        print("[Error] Failed to load saved data:", e)

  def loadSettings(self):
    if os.path.exists(SETTINGS_PATH):
      try:
        with open(SETTINGS_PATH, "r") as f:
          self.settings = json.load(f)
        print(f"[System] Settings loaded from the previous session.")
      except Exception as e:
        print("[Error] Failed to load saved settings:", e)
    else:
      print("[System] Using default settings.")

  def saveData(self):
    try:
      with open(DATA_PATH, "w") as f:
        json.dump(self.data, f, indent=2)
      print("\n[System] Data saved successfully.\n")
    except Exception as e:
      print("\n[Error] Failed to save data:", e)

  def saveSettings(self):
    try:
      with open(SETTINGS_PATH, "w") as f:
        json.dump(self.settings, f, indent=2)
      print("\n[System] Settings saved successfully.")
    except Exception as e:
      print("\n[Error] Failed to save settings:", e)

  def run(self):
    while True:
      if self.settings["clear_screen"] and self.needs_clear:
        self.clear()
        self.needs_clear = False

      print(r"""
  ____  ____  _____    _    __  __ ____
 |  _ \|  _ \| ____|  / \  |  \/  / ___|
 | | | | |_) |  _|   / _ \ | |\/| \___ \
 | |_| |  _ <| |___ / ___ \| |  | |___) |
 |____/|_| \_\_____/_/   \_\_|  |_|____/ """)

      self.divider()
      print("Select an option:")
      print("[1] Add new victim data")
      print("[2] View all data")
      print("[3] Edit existing data")
      print("[4] Delete an entry")
      print("[5] Settings")
      print("[6] Exit")

      choice = input("\n[System] Enter your choice: ").strip()

      match choice:
        case "1":
          self.addVictim()
        case "2":
          self.viewData()
        case "3":
          self.editData()
        case "4":
          self.deleteData()
        case "5":
          self.showSettings()
        case "6":
          self.saveSettings()
          self.saveData()
          print("[System] Exiting...")
          time.sleep(2)
          self.clear()
          break
        case _:
          print("\n[Error] Invalid option.\n")
          input("[System] Press Enter to continue...")
          self.needs_clear = True
          continue

      input("\n[System] Press Enter to return to menu...")
      self.needs_clear = True

  def showSettings(self):
    self.divider()
    print("Settings Menu:")
    print(f"[1] Toggle Clear Screen Mode ({'🟢' if self.settings['clear_screen'] else '🔴'})")
    print(f"[2] Configure Casualty Collection Point settings (No. teams = {self.settings['CCP_config'][0]}, Capacity = {'Infinity' if self.settings['CCP_config'][1] == 0 else self.settings['CCP_config'][1]})")
    print(f"[3] Configure Staging Area settings (No. teams = {self.settings['SA_config'][0]}, Capacity = {'Infinity' if self.settings['SA_config'][1] == 0 else self.settings['SA_config'][1]})")

    choice = input("\n[System] Enter your choice: ").strip()

    match choice:
      case "1":
        self.settings["clear_screen"] = not self.settings["clear_screen"]
        print(f"\nClear Screen Mode is now {'ON' if self.settings['clear_screen'] else 'OFF'}.")
      case "2":
        try:
          input1 = int(input("\n[Casualty Collection Point] Enter the number of first aid teams (positive number): "))

          if input1 > 0:
            self.settings["CCP_config"][0] = input1
          else:
            print("\n[Error] Number of teams must be a positive number. Keeping original setting.")
        except ValueError:
          print("\n[Error] Invalid input. Keeping original setting.")

        try:
          input2 = int(input("\n[Casualty Collection Point] Enter the maximum casualty capacity (nonnegative number, 0 for infinity): "))
          if input2 >= 0:
            self.settings["CCP_config"][1] = input2
          else:
            print("\n[Error] Capacity must be a nonnegative number. Keeping original setting.")
        except ValueError:
          print("\n[Error] Invalid input. Keeping original setting.")
      case "3":
        try:
          input1 = int(input("\n[Staging Area] Enter the number of transport teams (positive number): "))

          if input1 > 0:
            self.settings["SA_config"][0] = input1
          else:
            print("\n[Error] Number of teams must be a positive number. Keeping original setting.")
        except ValueError:
          print("\n[Error] Invalid input. Keeping original setting.")
        try:
          input2 = int(input("\n[Staging Area] Enter the maximum casualty capacity (nonnegative number, 0 for infinity): "))
          if input2 >= 0:
            self.settings["SA_config"][1] = input2
          else:
            print("\n[Error] Capacity must be a nonnegative number. Keeping original setting.")
        except ValueError:
          print("\n[Error] Invalid input. Keeping original setting.")
    self.saveSettings()
    print("")

  def addVictim(self):
    self.divider()
    print("[New Entry]\n")
    entry = {
      "SAR dispatched": self.askTime("Enter time SAR dispatched to Hot Zone: "),
      "SAR returned": self.askTime("Enter time of victim's arrival to CCP / SAR dispatched again: "),
      "First aid returned": self.askTime("Enter time victim's first aid team returned to post: "),
      "SA arrival": self.askTime("Enter time victim arrived to Staging Area: "),
      "Transport returned": self.askTime("Enter time transport unit returned from AMP: ")
    }
    self.data.append(entry)
    print("\nVictim data added successfully.")
    self.computeAverages(False)
    self.saveData()

  def viewData(self):
    self.divider()

    if not self.data:
      print("No data available.")
      return

    for i, entry in enumerate(self.data, 1):
      print(f"[Victim {i}]")
      for key, value in entry.items():
        print(f"  {key}: {value}")
      print("")

    self.computeAverages()

  def editData(self):
    self.viewData()

    if not self.data:
      return

    try:
      index = int(input("\n[System] Enter the victim number to edit: ")) - 1
      if 0 <= index < len(self.data):
        entry = self.data[index]
        print("\n[System] Editing entry. Press Enter to keep current value.")
        for key in entry:
          newVal = input(f"{key} [{entry[key]}]: ").strip()
          if newVal:
            if self.validateTime(newVal):
              entry[key] = newVal
            else:
              print(f"[Error] Invalid format. Keeping original {key}.")
        print("\n[System] Entry updated.")
        self.computeAverages(False)
        self.saveData()
      else:
        print("\n[Error] Invalid victim number.")
    except ValueError:
      print("\n[Error] Invalid input.")

  def deleteData(self):
    self.viewData()

    if not self.data:
      return

    try:
      index = int(input("\n[System] Enter the victim number to delete: ")) - 1
      if 0 <= index < len(self.data):
        del self.data[index]
        print("\n[System] Entry deleted.")
        self.computeAverages(False)
        self.saveData()
      else:
        print("\n[Error] Invalid victim number.")
    except ValueError:
      print("\n[Error] Invalid input.")

  def computeAverages(self, show = True):
    CCPArrivalRates, CCPServiceRates, SAServiceRates = [], [], []
    for entry in self.data:
      CCPArrivalRates.append(self.getRate(entry["SAR dispatched"], entry["SAR returned"]))
      CCPServiceRates.append(self.getRate(entry["SAR returned"], entry["First aid returned"]))
      SAServiceRates.append(self.getRate(entry["SA arrival"], entry["Transport returned"]))
    if CCPArrivalRates:
      self.CCPArrivalRateAverage = math.ceil(sum(CCPArrivalRates) / len(CCPArrivalRates))
      self.CCPServiceRateAverage = math.ceil(sum(CCPServiceRates) / len(CCPServiceRates))
      self.SAServiceRateAverage = math.ceil(sum(SAServiceRates) / len(SAServiceRates))
      if show:
        print(f"Average search and rescue rate = {self.CCPArrivalRateAverage} casualties / hour")
        print(f"Average first aid treatment rate = {self.CCPServiceRateAverage} casualties / hour")
        print(f"Average casualty transport rate = {self.SAServiceRateAverage} casualties / hour")

  def askTime(self, prompt):
    while True:
      t = input(prompt)
      if self.validateTime(t): return t
      print("\n[Error] Invalid time format. Use HHmm (24-hour, e.g. 0900).\n")

  def validateTime(self, time):
    return bool(re.fullmatch(r'^$|^([01]\d|2[0-3])[0-5]\d$', time.strip())) # For HH:mm, bool(re.fullmatch(r'^([01]?[0-9]|2[0-3]):[0-5][0-9]$', time.strip()))

  def getRate(self, t1, t2):
    if not t1 or not t2:
        return 0

    h1, m1 = int(t1[:2]), int(t1[2:])
    h2, m2 = int(t2[:2]), int(t2[2:])
    diffMins = abs((h2 * 60 + m2) - (h1 * 60 + m1))

    if diffMins == 0:
        return 0
    return (1 / diffMins) * 60

    # For HH:mm,
    # h1, m1 = map(int, t1.split(":"))
    # h2, m2 = map(int, t2.split(":"))
    # return (1 / abs((h2*60 + m2) - (h1*60 + m1))) * 60

  def clear(self):
    time.sleep(0.1)
    clear_output(wait=False)
    time.sleep(0.01)

  def divider(self):
    print("\n" + "-" * 60 + "\n")

calculator = CasualtyCalculator()
calculator.run()

In [ ]:
# @title $\text{[B] Model}$

import pandas as pd
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as tick
from IPython.display import display, HTML

def main() -> None:
  try:
    calculator
  except NameError:
    print("[Error] No database found. Please run Section [A] first.")
    return

  print("[Instructions] Provide what is being asked. Please refer to the User Manual if needed.\n")

  stages = ["Casualty Collection Point", "Staging Area"] # ["Casualty Collection Point", "Staging Area", "Advanced Medical Post"]
  dataFramesSummary = []
  dataFramesProbabilities = []
  timePerStage = []
  throughputPerStage = []
  recoPerStage = []
  interpPerStage = []
  prev_μ = None

  for stage in stages:
    if stage == "Casualty Collection Point":
      λ = calculator.CCPArrivalRateAverage #promptInteger("positive", f"Enter search and rescue rate per hour ({stage}): \n", "\nError: Please enter a positive number.")
      μ = calculator.CCPServiceRateAverage #promptInteger("positive", f"Enter first aid treatment rate per hour ({stage}): \n", "\nError: Please enter a positive number.")
      c = calculator.settings["CCP_config"][0]
      N = calculator.settings["CCP_config"][1]
    elif stage == "Staging Area":
      λ = prev_μ
      μ = calculator.SAServiceRateAverage #promptInteger("positive", f"Enter transport rate per hour ({stage}): \n", "\nError: Please enter a positive number.")
      c = calculator.settings["SA_config"][0]
      N = calculator.settings["SA_config"][1]
    else:
      λ = prev_μ
      μ = int(input(f"Enter medical treatment rate per hour ({stage}): \n"))
    # c : int = promptInteger("positive", f"Enter number of teams ({stage}): \n", "\nError: Please enter a positive number.")
    # N : int = promptInteger("nonnegative", f"Enter system capacity (0 for infinite) ({stage}): \n", "\nError: Please enter a nonnegative number.")

    if (λ == 0 or λ == None) or (μ == 0 or μ == None):
      print(f"[Error] Not enough data for {stage}. Make sure data is complete for this stage.\n")
      continue

    prev_μ = μ
    ρ = λ / μ

    if N > 0 and c > N:
      print(f"[Error] Number of teams ({c}) must be less than or equal to system capacity ({N}) for {stage}. To continue, you must change the settings.")
      while c > N:
        c = promptInteger("positive", f"Enter number of teams ({stage}): \n", "\nError: Please enter a positive number.")
        if c <= N:
          break
        else:
          print(f"[Error] Number of teams ({c}) must be less than or equal to system capacity ({N}) for {stage}.")
      if stage == "Casualty Collection Point":
        calculator.settings["CCP_config"][0] = c
      elif stage == "Staging Area":
        calculator.settings["SA_config"][0] = c
      calculator.saveSettings()
      print("")

    if N == 0 and ((c == 1 and not ρ < 1) or (c >= 2 and not (ρ / c) < 1)):
      c, N = resolveUnstability(stage, ρ)
      print("")

    stageResult, stageReco = summary(stage, ρ, λ, μ, c, N)
    stageTime = float(stageResult.loc[stageResult["Performance Metric"] == "Expected total time a casualty spends in the system", " "].values[0])
    stageLambdaEff = float(stageResult.loc[stageResult["Parameter"] == "Effective Casualty Arrival Rate", ""].values[0])
    stageServiceTimesTeams = float(stageResult.loc[stageResult["Parameter"] == "Service Rate", ""].values[0]) * float(stageResult.loc[stageResult["Parameter"] == "Number of teams", ""].values[0])
    stageThroughput = min(stageLambdaEff, stageServiceTimesTeams)
    timePerStage.append(stageTime)

    if stage == "Casualty Collection Point":
      if stageThroughput == stageLambdaEff:
        throughputPerStage.append([stageThroughput, "casualties / hour (Casualty Collection Point, Search and Rescue Rate)"])
      else:
        throughputPerStage.append([stageThroughput, "casualties / hour (Casualty Collection Point, First Aid Treatment Rate)"])
    elif stage == "Staging Area":
      throughputPerStage.append([stageThroughput, "casualties / hour (Staging Area, Transport Rate)"])

    dataFramesSummary.append(formatValues(stageResult))
    recoPerStage.append(stageReco)

    probResult, stageInterp = p(stage, ρ, c, N)
    dataFramesProbabilities.append((stage, probResult))
    interpPerStage.append(stageInterp)

    print("")

  if not dataFramesSummary:
    print("[Error] No data found for each stage. Please input data using Section [A].")
    return

  print("Summary of Results:\n")
  combinedSummary(stages, dataFramesSummary, dataFramesProbabilities, recoPerStage, interpPerStage)


  print("\nAnalysis: \n")
  M = int(input("[System] Enter targeted number of casualties: \n"))
  print(f"\nExpected total response time for {M} casualties: {M * sum(timePerStage)} hours")

  maxThroughput = min([throughputPerStage[i][0] for i in range(len(throughputPerStage))])
  for i in range(len(throughputPerStage)):
    if throughputPerStage[i][0] == maxThroughput:
      print(f"Effective response rate of the whole system (bottleneck): {maxThroughput} {throughputPerStage[i][1]}")

def summary(text : str, ρ : float, λ : int, μ : int, c : int, N : int = 0) -> pd.DataFrame:
  if N == 0:
    λ_eff = λ
    λ_lost = 0
    if c == 1:
      L_s = (ρ) / (1 - ρ)
      L_q = (ρ**2) / (1 - ρ)
      W_s = 1 / (μ * (1 - ρ))
      W_q = ρ / (μ * (1 - ρ))
    else:
      p0 = (sum([(ρ**n) / math.factorial(n) for n in range(c)]) + (((ρ**c) / math.factorial(c)) * (1 / (1 - (ρ / c)))))**(-1)
      L_q = ((ρ**(c + 1)) / (math.factorial(c - 1) * (c - ρ)**2)) * p0
      L_s = L_q + ρ
      W_s = L_s / λ_eff
      W_q = L_q / λ_eff
  else:
    ρ_c = ρ / c
    if ρ_c != 1:
      p0 = (sum([(ρ**n) / math.factorial(n) for n in range(c)]) + (((ρ**c)*(1 - (ρ_c)**(N - c + 1)))/(math.factorial(c) * (1 - ρ_c))))**(-1)
      L_q = (((ρ**(c + 1)) / (math.factorial(c - 1) * (c - ρ)**(2))) * (1 - ((ρ_c)**(N - c + 1)) - (N - c + 1) * (1 - ρ_c) * (ρ_c)**(N - c))) * p0
    else:
      p0 = (sum([(ρ**n) / math.factorial(n) for n in range(c)]) + ((ρ**c)/(math.factorial(c))) * (N - c + 1))**(-1)
      L_q = (((ρ**c)*(N - c)*(N - c + 1))/(2 * math.factorial(c))) * p0
    pN = ((ρ**N) / (math.factorial(c) * (c**(N - c)))) * p0
    λ_lost = λ * pN
    λ_eff = λ - λ_lost
    L_s = L_q + (λ_eff / μ)
    W_s = L_s / λ_eff
    W_q = L_q / λ_eff

  c_bar = L_s - L_q
  percent_util = (c_bar / c) * 100

  data = {
    "Parameter": [
        "Casualty Arrival Rate",
        "Effective Casualty Arrival Rate",
        "Effective and Actual Casualty Arrival Rate Difference",
        "Service Rate",
        "Average proportion of time teams are busy",
        "Number of teams"
    ],
    "": [λ, λ_eff, λ_lost, μ, ρ / c, c],
    "Performance Metric": [
        "Expected number of casualties in the system (queue + service) ",
        "Expected number of casualties waiting in the queue (not being served)",
        "Expected total time a casualty spends in the system",
        "Expected waiting time in the queue before being served",
        "Expected number of busy teams",
        "Percent team utilization"
    ],
    " ": [L_s, L_q, W_s, W_q, c_bar, percent_util]
  }

  df = pd.DataFrame(data)
  print(f"Parameters and Performance Metrics of the (M/M/{c}):(GD/{N if N > 0 else '∞'}/∞) model:\n")

  styled_df = styled(df, text)
  display(styled_df)

  reco = recommendations(percent_util, text)
  print(reco)

  return df, reco

def p(text: str, ρ: float, c: int, N: int = 0):
  table = []
  cprob = 0
  df = None

  if N == 0:
    if c == 1:
      i = 0
      while abs(cprob - 1) > 1e-6:
          prob = (1 - ρ)*(ρ**i)
          cprob += prob
          table.append([i, prob, cprob])
          i += 1
      df = pd.DataFrame(table, columns = ["n", "prob", "cprob"])
    else:
      p0 = (sum([(ρ**n) / math.factorial(n) for n in range(c)]) + (((ρ**c) / math.factorial(c)) * (1 / (1 - (ρ / c)))))**(-1)
      cprob += p0
      table.append([0, p0, cprob])

      i = 1
      max_iterations = N + 10 if N > 0 else 1000
      while i <= max_iterations and abs(cprob - 1) > 1e-6:
        prob = 0
        if i < c:
          prob = ((ρ**i) / math.factorial(i)) * p0
        elif i >= c:
          prob = ((ρ**i) / (math.factorial(c) * (c**(i - c)))) * p0
        cprob += prob
        table.append([i, prob, cprob])
        i += 1
      df = pd.DataFrame(table, columns = ["n", "prob", "cprob"])
  else:
    p_c = ρ / c
    if p_c != 1:
      p0 = (sum([(ρ**n) / math.factorial(n) for n in range(c)]) + (((ρ**c)*(1 - (p_c)**(N - c + 1)))/(math.factorial(c) * (1 - p_c))))**(-1)
    else:
      p0 = (sum([(ρ**n) / math.factorial(n) for n in range(c)]) + ((ρ**c)/(math.factorial(c))) * (N - c + 1))**(-1)
    cprob += p0
    table.append([0, p0, cprob])

    i = 1
    while i <= N:
        prob = 0
        if i < c:
          prob = ((ρ**i) / math.factorial(i)) * p0
        elif i >= c:
          prob = ((ρ**i) / (math.factorial(c) * (c**(i - c)))) * p0
        cprob += prob
        table.append([i, prob, cprob])
        i += 1
    df = pd.DataFrame(table, columns = ["n", "prob", "cprob"])

  print(f"\nProbabilities of the (M/M/{c}:GD/{N if N > 0 else '∞'}/∞) model:\n")

  n_values = df["n"]
  probabilities = df["prob"]

  fig = plt.figure(figsize=(10, 5))

  markerline, stemlines, baseline = plt.stem(
      n_values,
      probabilities,
      linefmt='C0-',
      markerfmt='C0o',
      basefmt=" "
  )

  plt.setp(markerline, markersize=8, markeredgewidth=1.5)
  plt.setp(stemlines, linewidth=1.5)

  plt.xlabel('Number of Casualties (n)', fontsize=12, labelpad=10)
  plt.ylabel('Probability (%)', fontsize=12, labelpad=10)
  plt.title(f'Probability Distribution of Casualty Counts in {text}', fontsize=14, pad=20)

  plt.xticks(n_values[::math.ceil(len(n_values) / 40)], fontsize=10, rotation = 45)
  plt.yticks(fontsize=10)
  plt.grid(axis='y', linestyle='--', alpha=0.7)

  plt.gca().spines[['top', 'right']].set_visible(False)
  plt.gca().yaxis.set_major_formatter(tick.PercentFormatter(xmax=1.0))

  plt.tight_layout()
  plt.show()

  topN = df.sort_values(by="prob", ascending=False).head(3)
  mostLikelyN = topN.iloc[0]
  modeN = int(mostLikelyN["n"])
  probN = mostLikelyN["prob"]

  lowCongestionMax = max(2, c)
  cumulativeLow = df[df["n"] <= lowCongestionMax]["prob"].sum()
  overloadThreshold = min(N, c + 3) if N > 0 else c + 3
  probOverload = df[df["n"] > overloadThreshold]["prob"].sum()

  interps = interpretations(modeN, probN, cumulativeLow, lowCongestionMax, probOverload, overloadThreshold, text)
  print(interps)

  return df, interps

def combinedSummary(stages, dataFramesSummary, dataFramesProbabilities, recommendations, interpretations) -> None:
  base = dataFramesSummary[0][["Parameter", "Performance Metric"]].copy()

  for i, stageDf in enumerate(dataFramesSummary):
      stageName = stages[i]
      stageVal = stageDf[["", " "]].copy()
      stageVal.columns = [f"{stageName} (Param)", f"{stageName} (Perf)"]
      base = pd.concat([base, stageVal], axis=1)

  dfColumns = list(base.columns)
  paramCols = [col for col in dfColumns if col.endswith("(Param)")]
  perfCols = [col for col in dfColumns if col.endswith("(Perf)")]
  columnOrder = ['Parameter'] + paramCols + ['Performance Metric'] + perfCols
  base = base[columnOrder]

  columnCleaned = []
  for col in base.columns:
      if "(Param)" in col or "(Perf)" in col:
          columnCleaned.append(col.replace(" (Param)", "").replace(" (Perf)", ""))
      else:
          columnCleaned.append(col)

  base.columns = columnCleaned
  baseRaw = base.reset_index(drop=True)
  html = baseRaw.to_html(index=False)

  display(HTML(html))


  for reco in recommendations:
    print(reco)

  print("")

  plt.figure(figsize=(12, 6))

  stageColors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
  max_n = []
  for i, (stageName, df) in enumerate(dataFramesProbabilities):
      n = df["n"]
      if len(n) > len(max_n):
        max_n = df["n"]
      prob = df["prob"]

      markerline, stemlines, baseline = plt.stem(
          n,
          prob,
          label = stageName,
          basefmt = " ",
      )

      plt.setp(markerline, color = stageColors[i % len(stageColors)], markersize=6, markeredgewidth=1.5,alpha = 0.7)
      plt.setp(stemlines, color = stageColors[i % len(stageColors)], linewidth=1.5, alpha=0.7)

  plt.xlabel('Number of Casualties (n)', fontsize=12)
  plt.ylabel('Probability (%)', fontsize=12)
  plt.title('Combined Probability Distribution of Casualty Counts', fontsize=14, pad=20)

  plt.xticks(max_n[::math.ceil(len(max_n) / 40)], fontsize=10, rotation = 45)
  plt.yticks(fontsize=10)

  plt.grid(axis='y', linestyle='--', alpha=0.7)
  plt.gca().spines[['top', 'right']].set_visible(False)
  plt.gca().yaxis.set_major_formatter(tick.PercentFormatter(xmax=1.0))
  plt.legend()
  plt.tight_layout()
  plt.show()

  for interp in interpretations:
    print(interp)

  print("")

def recommendations(util: float, stage: str = "") -> None:
  lines = []
  if stage:
    lines.append(f"\n[Recommendations for {stage}]")

    if util < 30:
      lines.append(f"• Utilization is very low ({util:.2f}). Teams are underutilized, leading to possible resource wastage.")
      lines.append("  ↪ Recommendation: Consider reducing the number of teams or reallocating them to other tasks.")
    elif 30 <= util < 60:
      lines.append(f"• Utilization is moderate ({util:.2f}). Teams are available, but there's potential for better efficiency.")
      lines.append("  ↪ Recommendation: Monitor other stages' stability. If delays are minimal, the setup is okay. If not, consider reallocating some teams to other stages.")
    elif 60 <= util <= 85:
      lines.append(f"• Utilization is within the optimal range ({util:.2f}).")
      lines.append("  ↪ Recommendation: No immediate action needed. Keep monitoring performance during peak loads.")
    elif 85 < util <= 95:
      lines.append(f"• High utilization. Teams are close to maximum capacity ({util:.2f}).")
      lines.append("  ↪ Recommendation: Consider adding more teams or increasing service rate if delay is observed.")
    else:
      lines.append(f"• Critical utilization! The system is at risk of becoming unstable ({util:.2f}).")
      lines.append("  ↪ Recommendation: Immediate intervention needed. Increase team count or expand capacity.")

  return "\n".join(lines)

def interpretations(modeN: float, probN: float, cumulativeLow: float, lowCongestionMax: float, probOverload: float, overloadThreshold: float, stage: str = "") -> None:
  lines = []

  if stage:
    lines.append(f"\n[Interpretation of Probability Distribution for {stage}]")
    lines.append(f"• The most likely number of casualties in the system in the long run is {modeN} "
        f"(occurring approximately {probN:.1%} of the time).")

    lines.append(f"• There's a {cumulativeLow:.1%} chance that the system has 0 to {lowCongestionMax} casualties at any moment, "
          f"which suggests a relatively low congestion under current conditions.")

    if probOverload > 0.1:
        lines.append(f"• Warning: There's a {probOverload:.1%} chance of having more than {overloadThreshold} casualties "
              "in the system. This may indicate potential strain or under-capacity.")
    else:
        lines.append(f"• Only {probOverload:.1%} of the time do we see more than {overloadThreshold} casualties, "
              "indicating a well-managed system.")

    return "\n".join(lines)

def resolveUnstability(stage, ρ):
    config = calculator.settings["CCP_config"] if stage == "Casualty Collection Point" else calculator.settings["SA_config"]
    c, N = config

    if N == 0:
        N = float("inf")

    minTeamsRequired = math.ceil(ρ) + 1

    print(f"\n[Error] The current setup for {stage} is unstable. This means that the teams can't keep up with incoming casualties.\n")
    print("Please choose how you'd like to fix it:")
    print("[1] Change the number of teams responding")
    print("[2] Set the maximum number of casualties the area can hold")
    print("[3] Let the system fix it for me (recommended)")

    while True:
        choice = input("\n[System] Your choice: ").strip()
        if choice == '1':
            while True:
                c = promptInteger("positive", f"Enter number of teams ({stage}) (must be at least {minTeamsRequired}): ",
                                  "\n[Error] Please enter a positive number.")
                if c < minTeamsRequired:
                    print(f"\n[Error] Number of teams must be at least {minTeamsRequired} to stabilize the system.")
                    continue
                if N != float("inf") and c > N:
                    print(f"\n[Error] Teams ({c}) must not exceed system capacity ({N}).")
                    continue
                break
        elif choice == '2':
            while True:
                N = promptInteger("positive", f"Enter system capacity ({stage}): ",
                                  "\n[Error] Please enter a positive number.")
                if c > N:
                    print(f"\nError] Teams ({c}) must not exceed new system capacity ({N}).")
                    continue
                break
        elif choice == '3':
            if N == float("inf"):
                N = math.ceil(ρ * 1.5)
            c = max(minTeamsRequired, min(math.ceil(ρ + 1), N))
            print(f"\n[System] Adjusted to {c} team(s) and capacity {N} for {stage}.")
        else:
            print("\n[Error] Invalid choice. Please select 1, 2, or 3.")
            continue
        break

    if N == float("inf"):
        N = 0

    if stage == "Casualty Collection Point":
        calculator.settings["CCP_config"] = [c, N]
    else:
        calculator.settings["SA_config"] = [c, N]

    calculator.saveSettings()
    return c, N

def promptInteger(condition: str, promptText: str, errorConditionText: str, valueErrorText: str = "\nError: Please enter a number.") -> int:
  while True:
    try:
        num = int(input(promptText))
        if condition == "positive":
          if num > 0:
            print("")
            break
          else:
            print(errorConditionText)
        if condition == "nonnegative":
          if num >= 0:
            print("")
            break
          else:
              print(errorConditionText)
    except ValueError:
        print(valueErrorText)
  return num

def styled(df: pd.DataFrame, text: str):
  styledDf = formatValues(df)
  return styledDf.style.set_caption(text)\
      .hide(axis="index")\
      .set_table_styles([{
          "selector": "th",
          "props": [("border", "1px solid gray"), ("font-weight", "bold")]
      }])\
      .set_properties(**{"border": "1px solid gray"})

def formatValues(df: pd.DataFrame) -> pd.DataFrame:
  params = df["Parameter"].tolist()
  perfs = df["Performance Metric"].tolist()
  paramVals = df[""].tolist()
  perfVals = df[" "].tolist()

  def formatParam(val: float, param: str):
    if "Arrival Rate" in param or "Service Rate" in param:
      return f"{val:.2f} casualties/hour"
    elif "of teams" in param:
      return f"{val:.2f} teams"
    else:
      return f"{val:.2f}"

  def formatPerf(val: float, metric: str):
    if "Percent" in metric:
      return f"{val:.2f}%"
    elif "casualties" in metric:
      return f"{val:.2f} casualties"
    elif "time" in metric:
      return f"{val * 60:.2f} minutes"
    elif "teams" in metric:
      return f"{val:.2f} teams"
    else:
      return f"{val:.2f}"

  formattedDf = df.copy()
  formattedDf[""] = [formatParam(val, param) for val, param in zip(paramVals, params)]
  formattedDf[" "] = [formatPerf(val, metric) for val, metric in zip(perfVals, perfs)]

  return formattedDf

def displayLabels(strList, initialSpace: int = 8, spaceBetween: int = 32) -> str:
  currentStr = ""
  space = initialSpace
  for i in range(0, len(strList)):
    if i == 0:
      currentStr = " " * space + strList[i]
      space = spaceBetween
      continue
    currentStr = currentStr + " " * space  + strList[i]
  return currentStr

if __name__ == "__main__":
    main()